# GeoMx spatial transcriptomics — S100P and MHC correlation analysis

**Goal:** Characterize the relationship between S100P expression and MHC class I/II
gene expression in LUAD ROIs from GeoMx spatial transcriptomics. Mutual information
is used as the primary association metric to capture non-linear dependence. Spearman
correlation is provided as a supplementary view. KRT genes serve as negative controls.

**Input:** GeoMx expression matrix and sample groupings CSV.

**Output:** Figure 4A (S100P–MHC MI scatter), Figure S5B (Spearman r), Figure S5C
(KRT negative control); supplementary correlation tables for S100P and HLA-DRA.

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.feature_selection import mutual_info_regression

sns.set(font_scale=1.5)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure4'
supp_out  = fig_dir   / 'figure4-supp'
table_out = table_dir / 'figure4'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

# MHC class I and II genes for main figure and S5B
mhc_gene_list = [
    'HLA-DRA', 'HLA-DRB1', 'HLA-DPA1', 'HLA-DPB1',
    'HLA-DQA1', 'HLA-DQB1', 'CD74', 'CIITA',
    'NLRC5', 'HLA-A', 'HLA-B', 'HLA-C',
]

# KRT negative controls for S5C
krt_gene_list = ['KRT8', 'KRT19', 'KRT31', 'KRT34', 'KRTDAP', 'KRT5']

# per-patient marker styles
marker_styles = ['o', 's', '^', 'D', 'v', 'P', 'X', 'h', '8', '*']

## Load data

GeoMx expression matrix (genes × ROIs) is transposed to ROIs × genes.
Sample groupings provide patient ID, MHC classification, and tissue type.
Analysis is restricted to LUAD ROIs (`type == 'AD'`). Patients are mapped
to anonymous letter codes for display.

In [ ]:
data_path   = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/geomx/data/spatial_transcriptomics.csv')
groups_path = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/geomx/data/geomx_sample_groupings_plsda.csv')

data   = pd.read_csv(data_path, index_col=0).T
groups = pd.read_csv(groups_path, converters={i: str for i in range(100)})
groups = groups[['sample name', 'groupings', 'type', 'Patient no', 'MHC1 score']]
groups = groups.set_index('sample name')

print(f"Expression matrix: {data.shape[0]:,} ROIs × {data.shape[1]:,} genes")
print(f"Sample groupings: {len(groups):,} samples")

## Patient metadata

GeoMx LUAD patients are identified by letter code in the groupings file.
A metadata table linking letter codes to cohort patient IDs, MHC1 status,
and GeoMx ROI sample numbers is saved as a supplementary table.

In [ ]:
# ROI to cohort ID mapping from study notes
roi_to_cohort = {}
for cohort_id, rois in [
    ('A57', [36,37,38,39,40,41]),
    ('A9',  [42,43,44]),
    ('A5',  [47,48,51,52]),
    ('A17', [53,54,55,56,57,58]),
    ('A22', [59,60,61,62]),
    ('A44', [65,67]),
]:
    for roi in rois:
        roi_to_cohort[roi] = cohort_id

# filter to LUAD
luad_groups = groups[groups['type'] == 'AD'].copy()

# extract ROI number
luad_groups['roi_number'] = (
    luad_groups.index.str.strip("'")
    .str.extract(r'_0*(\d+)_FullROI')
    .iloc[:, 0]
    .values
    .astype(int)
)

# map to cohort ID — explicit int cast on keys
luad_groups['cohort_id'] = luad_groups['roi_number'].map(
    {int(k): v for k, v in roi_to_cohort.items()}
)

# assign display labels A-F
patient_order = luad_groups['Patient no'].unique()
display_map = {p: letter for p, letter in zip(patient_order, 'ABCDEF')}
luad_groups['geomx_luad_no'] = luad_groups['Patient no'].map(display_map)

# save metadata
patient_meta = (
    luad_groups.groupby('geomx_luad_no')
    .agg(
        cohort_id=('cohort_id', 'first'),
        geomx_patient_no=('Patient no', 'first'),
    )
    .reset_index()
    .sort_values('geomx_luad_no')
)

patient_meta.to_csv(table_out / 'geomx_patient_metadata.csv', index=False)
print(patient_meta.to_string(index=False))

In [ ]:
luad_expr = expr.loc[expr['type'] == 'AD'].copy()
luad_expr['patient_label'] = luad_expr['patient'].map(display_map)

print(f"LUAD ROIs: {len(luad_expr):,}")
print(f"Patients: {sorted(luad_expr['patient_label'].unique())}")

## Scatter plot function

Shared plotting function for Figures 4A, S5B, and S5C. Each panel shows
one gene vs S100P across all LUAD ROIs, with points colored by patient
using distinct markers. A regression line is overlaid. Annotation metric
(MI or Spearman r) is shown in the top-right corner of each panel.

In [ ]:
def plot_s100p_scatter(df, gene_list, annotation='mi', ncols=6, figsize=None,
                       fig_path=None, regression=True,
                       title_fontsize=11, label_fontsize=10,
                       tick_fontsize=9, annotation_fontsize=11):
    """
    Scatter plots of S100P vs each gene in gene_list across LUAD ROIs.

    Parameters
    ----------
    df : pd.DataFrame
        LUAD expression DataFrame with 'S100P', 'patient_label' columns
        and one column per gene.
    gene_list : list of str
        Genes to plot against S100P.
    annotation : str
        'mi' for mutual information, 'spearman' for Spearman r.
    ncols : int
        Number of columns in the figure grid.
    figsize : tuple or None
        Figure size. If None, inferred from ncols and nrows.
    fig_path : Path or None
        Output PDF path. If None, figure is not saved.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    nrows = int(np.ceil(len(gene_list) / ncols))
    if figsize is None:
        figsize = (ncols * 4, nrows * 3.5)

    df_clean = df.dropna(subset=['S100P'] + gene_list)
    patients = df_clean['patient_label'].unique()

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()

    for i, gene in enumerate(gene_list):
        ax = axes[i]

        # per-patient scatter
        for j, patient in enumerate(patients):
            subset = df_clean[df_clean['patient_label'] == patient]
            ax.scatter(
                subset['S100P'], subset[gene],
                marker=marker_styles[j % len(marker_styles)],
                color='black', s=50,
                label=patient if i == 0 else '',
            )

        # conditional regression line
        if regression:
            sns.regplot(
                x='S100P', y=gene, data=df_clean, ax=ax,
                scatter=False, color='black',
                line_kws={'linewidth': 2},
            )

        # annotation
        if annotation == 'mi':
            mi = mutual_info_regression(
                df_clean[['S100P']], df_clean[gene]
            )[0]
            label = f'MI = {mi:.2f}'
        else:
            rho, _ = stats.spearmanr(df_clean['S100P'], df_clean[gene])
            label = f'Spearman r = {rho:.2f}'

        ax.text(
            0.95, 0.95, label,
            transform=ax.transAxes,
            ha='right', va='top', fontsize=annotation_fontsize,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.5),
        )
        ax.set_xlabel('S100P', fontsize=label_fontsize)
        ax.set_ylabel(gene, fontsize=title_fontsize)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0)
        ax.tick_params(labelsize=tick_fontsize)
        sns.despine(ax=ax)

    # hide unused panels
    for ax in axes[len(gene_list):]:
        ax.set_visible(False)

    # shared patient legend from first panel
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='upper right', title='Patient',
        bbox_to_anchor=(1.0, 1.0),
        fontsize=label_fontsize,
    )

    plt.tight_layout(rect=[0, 0, 0.92, 1])

    if fig_path is not None:
        plt.savefig(fig_path, bbox_inches='tight')

    return fig

## Figure 4A — S100P vs MHC gene expression (mutual information)

Mutual information between S100P and MHC class I/II genes across LUAD ROIs.
Each point is one GeoMx ROI; regression line is fit across all ROIs pooled.
MI quantifies non-linear association without assuming a linear relationship.

In [ ]:
fig = plot_s100p_scatter(
    luad_expr, mhc_gene_list,
    annotation='mi',
    regression = False,
    ncols=6,
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=18, 
    annotation_fontsize=18,
    fig_path=fig_out / 'fig4a_geomx_s100p_mhc_mi.pdf',
)
plt.show()

## Figure S5B — S100P vs MHC gene expression (Spearman r)

Same gene set as Figure 4A, annotated with Spearman rank correlation
coefficient as a linear-rank complement to the mutual information view.

In [ ]:
fig = plot_s100p_scatter(
    luad_expr, mhc_gene_list,
    annotation='spearman',
    ncols=6,
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=18, 
    annotation_fontsize=14,
    fig_path=supp_out / 'figS5b_geomx_s100p_mhc_spearman.pdf',
)
plt.show()

## Figure S5C — S100P vs KRT genes (negative control)

Keratin genes are structural epithelial markers with no known MHC regulatory
relationship. Low MI values here contrast with the MHC gene panel and
support the specificity of the S100P–MHC II association.

In [ ]:
fig = plot_s100p_scatter(
    luad_expr, krt_gene_list,
    annotation='mi',
    ncols=6,
    regression= False,
    figsize=(24, 4),
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=18, 
    annotation_fontsize=18,
    fig_path=supp_out / 'figS5c_geomx_s100p_krt_negative_control.pdf',
)
plt.show()

In [ ]:
fig = plot_s100p_scatter(
    luad_expr, krt_gene_list,
    annotation='spearman',
    ncols=6,
    figsize=(24, 4),
    title_fontsize=18,
    label_fontsize=18,
    tick_fontsize=18, 
    annotation_fontsize=14,
    fig_path=supp_out / 'figS5c_geomx_s100p_krt_negative_control_spearman.pdf',
)
plt.show()

## Supplementary correlation tables

Pearson correlation of all genes with S100P and HLA-DRA across LUAD ROIs,
saved as supplementary CSV tables for reviewer reference.`

In [ ]:
# restrict to numeric gene columns only
gene_cols = luad_expr.columns.difference(['groupings', 'type', 'patient', 'patient_label'])
luad_numeric = luad_expr[gene_cols].apply(pd.to_numeric, errors='coerce')

# S100P correlates
s100p_corr = luad_numeric.corr()['S100P'].drop('S100P').sort_values(ascending=False)
s100p_corr_df = s100p_corr.reset_index()
s100p_corr_df.columns = ['gene', 'pearson_r']
s100p_corr_df.to_csv(table_out / 'geomx_s100p_correlates.csv', index=False)

# HLA-DRA correlates
hladra_corr = luad_numeric.corr()['HLA-DRA'].drop('HLA-DRA').sort_values(ascending=False)
hladra_corr_df = hladra_corr.reset_index()
hladra_corr_df.columns = ['gene', 'pearson_r']
hladra_corr_df.to_csv(table_out / 'geomx_hladra_correlates.csv', index=False)

print(f"S100P correlates saved: {len(s100p_corr_df):,} genes")
print(f"HLA-DRA correlates saved: {len(hladra_corr_df):,} genes")
print(f"\nTop 10 S100P positive correlates:")
print(s100p_corr_df.head(10).to_string(index=False))
print(f"\nTop 10 S100P negative correlates:")
print(s100p_corr_df.tail(10).sort_values('pearson_r').to_string(index=False))
print(f"\nTop 10 HLA-DRA positive correlates:")
print(hladra_corr_df.head(10).to_string(index=False))
print(f"\nTop 10 HLA-DRA negative correlates:")
print(hladra_corr_df.tail(10).sort_values('pearson_r').to_string(index=False))

## Interpretation

The GeoMx correlation analysis reveals a clear inverse relationship between
S100P and MHC II gene expression across LUAD ROIs. HLA-DPA1, HLA-DRB1,
HLA-DQB1, and HLA-DPB1 dominate the top negative correlates of S100P,
directly validating the S100P–MHC II axis at the RNA level in spatially
resolved tumor regions. ITGB2 and CLEC10A appearing among the negative
correlates further suggests that S100P-high regions have reduced immune
infiltration broadly, not just reduced MHC II expression.

On the HLA-DRA side, CD74 (r=0.987) is near-perfectly correlated —
consistent with their co-regulation as core MHC II machinery components.
S100P itself appears among the top HLA-DRA negative correlates (r=-0.718),
confirming the relationship is bidirectional and robust.

The S100P positive correlates are biologically provocative. PRDX1 and PRDX6
(peroxiredoxins), TALDO1 (transaldolase, pentose phosphate pathway), and
PSENEN (presenilin enhancer, involved in Notch/gamma-secretase signaling)
suggest S100P-high tumors may be operating under oxidative and metabolic
stress. S100P is a calcium-binding protein known to be induced by hypoxia
and metabolic reprogramming — the co-expression of these genes raises the
question of whether S100P upregulation is a downstream consequence of
metabolic stress rather than a primary driver, and whether this stress
state actively suppresses MHC II antigen presentation machinery.
This warrants further investigation and may connect to the hypoxic pathway
enrichment seen in the GSEA results.